In [7]:
from pathlib import Path

In [9]:
data_dir = Path("/home/rooneyish/Documents/Pulse_SLM/data")
train_file = data_dir / "train.txt"
val_file = data_dir / "val.txt"

In [11]:
train_txt = open(train_file).read()
val_txt = open(val_file).read()

In [22]:
import re

def clean_txt(text):
    # Remove the backslashes escaping the single quotes (\'s -> 's)
    text = text.replace("\\'", "'")
    
    # Fix the weird spacing around the text formatting
    text = re.sub(r"\s+'\s*([sn])", r"'\1", text)   # fixes ' s or ' n
    text = re.sub(r"={1,}\s*(.*?)\s*={1,}", r"\1", text) # removes headers
    text = re.sub(r"\s+([,.:;?!])", r"\1", text)    # fixes punctuation spacing
    text = re.sub(r"\s*@\s*-\s*@\s*", "-", text)    # fixes @-@ hyphens
    text = re.sub(r"\s+", " ", text).strip()        # collapses extra spaces
    
    return text

In [24]:
clean_train_txt = clean_txt(train_txt)
clean_val_txt = clean_txt(val_txt)

In [27]:
len(clean_train_txt), len(clean_val_txt)

(10521374, 1103839)

In [25]:
full_txt = clean_train_txt + clean_val_txt

In [26]:
len(full_txt)

11625213

In [30]:
import nltk.data

nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /home/rooneyish/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/rooneyish/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [31]:
from nltk.tokenize import sent_tokenize

sent_tokens = sent_tokenize(full_txt)

In [32]:
sent_tokens[:5]

['Valkyria Chronicles III Senjō no Valkyria 3: Unrecorded Chronicles ( Japanese: 戦場のヴァルキュリア3, lit.',
 'Valkyria of the Battlefield 3 ), commonly referred to as Valkyria Chronicles III outside Japan, is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable.',
 'Released in January 2011 in Japan, it is the third game in the Valkyria series.',
 'Employing the same fusion of tactical and real-time gameplay as its predecessors, the story runs parallel to the first game and follows the " Nameless ", a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven ".',
 'The game began development in 2010, carrying over a large portion of the work done on Valkyria Chronicles II.']

In [35]:
sent_w_SE = []
for each in sent_tokens:
    each = ''.join('<S> '+ each + ' <E>')
    sent_w_SE.append(each)

In [36]:
sent_w_SE[:5]

['<S> Valkyria Chronicles III Senjō no Valkyria 3: Unrecorded Chronicles ( Japanese: 戦場のヴァルキュリア3, lit. <E>',
 '<S> Valkyria of the Battlefield 3 ), commonly referred to as Valkyria Chronicles III outside Japan, is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable. <E>',
 '<S> Released in January 2011 in Japan, it is the third game in the Valkyria series. <E>',
 '<S> Employing the same fusion of tactical and real-time gameplay as its predecessors, the story runs parallel to the first game and follows the " Nameless ", a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven ". <E>',
 '<S> The game began development in 2010, carrying over a large portion of the work done on Valkyria Chronicles II. <E>']

In [47]:
tokens = []
for sentence in sent_w_SE:
    word = sentence.split()
    tokens.append(word)

In [48]:
len(tokens)

86251

In [65]:
corpus = {}
all_tokens = []
for each in tokens:
    for i in range(len(each)):
        all_tokens.append(each[i])

for each in all_tokens:
    if each in corpus:
        corpus[each] += 1
    else:
        corpus[each] = 1

corpus = {k:v for k, v in sorted(corpus.items(), key = lambda item: item[1], reverse = True)}
del corpus['<S>']
del corpus['<E>']
len(corpus)

134357

In [68]:
pad_token = "<PAD>"
unk_token = "<UNK>"
start_token = "<S>"
end_token = "<E>"

pad_id = 0
unk_id = 1
start_id = 2
end_id = 3

unique_words = list(corpus.keys())

In [76]:
stoi = {pad_token: pad_id, unk_token: unk_id, start_token: start_id, end_token: end_id}
itos = {pad_id: pad_token, unk_id: unk_token, start_id: start_token, end_id: end_token}

In [77]:
for idx, word in enumerate(unique_words):
    target_id = idx + 4
    stoi[word] = target_id
    itos[target_id] = word

In [79]:
len(stoi), len(itos)

(134361, 134361)

--- Tokenizer Diagnostics ---
Target Token List: ['<S>', 'valkyria', 'chronicles', 'iii', 'is', 'a', 'tactical', 'game', '<E>']
Numerical IDs:     [2, 1, 15149, 1, 20, 9, 7006, 117, 3]
Decoded Output:    ['<S>', '<UNK>', 'chronicles', '<UNK>', 'is', 'a', 'tactical', 'game', '<E>']
